# Penn Campus Style Transfer — Dual ControlNet (Canny + Segmentation) + LoRA

## Architecture
- **Base:** Stable Diffusion v1.5
- **Control:** Canny Edge ControlNet + Segmentation ControlNet (dual conditioning)
- **Adaptation:** LoRA fine-tuning on Penn-specific appearance
- **Inference:** img2img (SDEdit) + dual ControlNet + LoRA

## Why Canny + Segmentation?
- **Canny:** Preserves fine architectural details (windows, brick patterns, ornaments)
- **Segmentation:** Prevents semantic drift (sky stays sky, building stays building)
- **Together:** Best of both worlds for architectural scenes

## Pipeline
1. Use existing Canny maps (already generated)
2. Generate segmentation maps for all images
3. Train LoRA on SD + dual ControlNet
4. Inference with img2img + dual control + LoRA

**Run on:** Colab Pro+ A100 GPU

## Cell 1: Environment Setup

In [1]:
# Install dependencies
!pip install diffusers==0.31.0 transformers==4.44.0 peft==0.12.0 accelerate==0.34.0 torchao>=0.16.0 scikit-learn -q
!pip install controlnet-aux opencv-python -q

# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 1.8 MB/s eta 0:00:00
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB
Mounted at /content/drive


## Cell 2: Import Libraries

In [2]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

from diffusers import (
 StableDiffusionControlNetPipeline,
 StableDiffusionControlNetImg2ImgPipeline,
 ControlNetModel,
 DDPMScheduler,
 AutoencoderKL,
 UNet2DConditionModel,
 UniPCMultistepScheduler
)
from transformers import CLIPTextModel, CLIPTokenizer
from peft import LoraConfig, get_peft_model

device ="cuda"
dtype = torch.float16

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

## Cell 3: Path Configuration

In [10]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Google Drive paths
BASE_PATH ="/content/drive/MyDrive/5190 S26 Final Project"

# Input
IMAGE_DIR = f"{BASE_PATH}/Dataset/MODELIMG/Resized and cleaned"
CANNY_MAP_DIR = f"{BASE_PATH}/Dataset/MODELIMG/CannyMap"
SEG_MAP_DIR = f"{BASE_PATH}/Dataset/MODELIMG/SegmentationMap"
METADATA_FILE = f"{BASE_PATH}/Dataset/BoundaryMetadata.xlsx"

# Automatically generated splits
TRAIN_CSV_PATH = f"{BASE_PATH}/penn_lora_train.csv"
VAL_CSV_PATH = f"{BASE_PATH}/penn_lora_val.csv"

# Models & Outputs
# UPDATED: Replaced local Drive path with Hugging Face Repo ID
SD_PATH ="runwayml/stable-diffusion-v1-5"

LORA_OUTPUT_DIR = f"{BASE_PATH}/Dataset/LoRA_CannySegmentation_Checkpoints"
VALIDATION_DIR = f"{BASE_PATH}/Validation_CannySegmentation"

os.makedirs(LORA_OUTPUT_DIR, exist_ok=True)
os.makedirs(VALIDATION_DIR, exist_ok=True)

# Generate Train/Val Split Automatically
if os.path.exists(METADATA_FILE):
 print("Generating train/val splits from metadata...")

 # UPDATED: Use read_excel for .xlsx files (requires openpyxl)
 df_full = pd.read_excel(METADATA_FILE, sheet_name="Prompts and MD")

 # 80-20 Train-Val split
 train_df, val_df = train_test_split(df_full, test_size=0.2, random_state=42)

 train_df.to_csv(TRAIN_CSV_PATH, index=False)
 val_df.to_csv(VAL_CSV_PATH, index=False)

 print(f"Split complete: {len(train_df)} training samples, {len(val_df)} validation samples.")
else:
 print(f"Warning: Could not find {METADATA_FILE}. Ensure it is uploaded to your Drive.")

print("\nPaths configured:")
print(f"Canny maps: {CANNY_MAP_DIR} (already exists)")
print(f"Segmentation maps: {SEG_MAP_DIR} (already exists)")
print(f"Base Model: {SD_PATH} (will automatically download from Hugging Face)")

Generating train/val splits from metadata...
✓ Split complete: 245 training samples, 62 validation samples.

Paths configured:
Canny maps: /content/drive/MyDrive/5190 S26 Final Project/Dataset/MODELIMG/CannyMap (already exists)
Segmentation maps: /content/drive/MyDrive/5190 S26 Final Project/Dataset/MODELIMG/SegmentationMap (already exists)
Base Model: runwayml/stable-diffusion-v1-5 (will automatically download from Hugging Face)


## Cell 4: Generate Segmentation Maps

Canny maps already exist from your preprocessing — we just need to generate segmentation maps.

In [ ]:
print("Loading segmentation model...")
segmentor = OneFormerADE20kSegmentor.from_pretrained("lllyasviel/Annotators")

# Load image list from CSV
df = pd.read_csv(CSV_PATH)
image_files = df['filename'].tolist()

print(f"Generating segmentation maps for {len(image_files)} images...")

for filename in tqdm(image_files, desc="Segmentation"):
 input_path = os.path.join(IMAGE_DIR, filename)
 output_path = os.path.join(SEG_MAP_DIR, filename)

 # Skip if already exists
 if os.path.exists(output_path):
 continue

 try:
 image = Image.open(input_path).convert("RGB")
 seg_map = segmentor(image, detect_resolution=512, image_resolution=512)
 seg_map.save(output_path)
 except Exception as e:
 print(f"Error processing {filename}: {e}")

print("Segmentation maps generated.")
del segmentor
torch.cuda.empty_cache()

## Cell 5: Dataset Class for Canny + Segmentation

In [11]:
class PennCannySegDataset(Dataset):
"""
 Loads:
 - Image (for VAE encoding)
 - Canny edge map (for ControlNet 1 - fine detail)
 - Segmentation map (for ControlNet 2 - semantic regions)
 - Text prompt (for CLIP)
"""

 def __init__(self, csv_path, image_dir, canny_dir, seg_dir, tokenizer, resolution=512):
 self.df = pd.read_csv(csv_path)
 self.df = self._validate_files(self.df, image_dir, canny_dir, seg_dir)

 self.image_dir = image_dir
 self.canny_dir = canny_dir
 self.seg_dir = seg_dir
 self.tokenizer = tokenizer
 self.resolution = resolution

 # Image transform: normalize to [-1, 1] for VAE
 self.image_transform = transforms.Compose([
 transforms.Resize(resolution, interpolation=transforms.InterpolationMode.BILINEAR),
 transforms.CenterCrop(resolution),
 transforms.ToTensor(),
 transforms.Normalize([0.5], [0.5]),
 ])

 # Control map transform: normalize to [0, 1] for ControlNet
 self.control_transform = transforms.Compose([
 transforms.Resize(resolution, interpolation=transforms.InterpolationMode.BILINEAR),
 transforms.CenterCrop(resolution),
 transforms.ToTensor(),
 ])

 print(f"Dataset initialized with {len(self.df)} valid samples")

 def _validate_files(self, df, image_dir, canny_dir, seg_dir):
"""Drop rows where any required file is missing."""
 valid_rows = []
 missing_count = 0

 for idx, row in df.iterrows():
 filename = row['filename']

 # Try case variants
 candidates = [filename, filename.replace('.jpg', '.JPG'), filename.replace('.JPG', '.jpg')]

 found = False
 for candidate in candidates:
 img_path = os.path.join(image_dir, candidate)
 canny_path = os.path.join(canny_dir, candidate)
 seg_path = os.path.join(seg_dir, candidate)

 if os.path.exists(img_path) and os.path.exists(canny_path) and os.path.exists(seg_path):
 row['filename'] = candidate
 valid_rows.append(row)
 found = True
 break

 if not found:
 missing_count += 1
 print(f"Skipping missing file: {filename}")

 if missing_count > 0:
 print(f"Warning: {missing_count} files missing")

 return pd.DataFrame(valid_rows).reset_index(drop=True)

 def __len__(self):
 return len(self.df)

 def __getitem__(self, idx):
 row = self.df.iloc[idx]
 filename = row['filename']
 prompt = row['prompt']

 # Load image
 image = Image.open(os.path.join(self.image_dir, filename)).convert("RGB")
 pixel_values = self.image_transform(image)

 # Load Canny map (convert to RGB for ControlNet)
 canny_map = Image.open(os.path.join(self.canny_dir, filename)).convert("RGB")
 canny_values = self.control_transform(canny_map)

 # Load segmentation map (convert to RGB for ControlNet)
 seg_map = Image.open(os.path.join(self.seg_dir, filename)).convert("RGB")
 seg_values = self.control_transform(seg_map)

 # Tokenize prompt
 tokenized = self.tokenizer(
 prompt,
 max_length=self.tokenizer.model_max_length,
 padding="max_length",
 truncation=True,
 return_tensors="pt"
 )
 input_ids = tokenized.input_ids.squeeze(0)

 return {
"pixel_values": pixel_values,
"canny_values": canny_values,
"seg_values": seg_values,
"input_ids": input_ids,
"prompt": prompt,
 }


def collate_fn(batch):
"""Stack tensors, keep prompts as list."""
 return {
"pixel_values": torch.stack([b["pixel_values"] for b in batch]),
"canny_values": torch.stack([b["canny_values"] for b in batch]),
"seg_values": torch.stack([b["seg_values"] for b in batch]),
"input_ids": torch.stack([b["input_ids"] for b in batch]),
"prompts": [b["prompt"] for b in batch],
 }

print("Dataset class defined.")

Dataset class defined.


## Cell 6: Load Models

In [12]:
import torch
from transformers import CLIPTokenizer, CLIPTextModel
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler, ControlNetModel

# 1. Define your hardware and precision settings
device ="cuda"if torch.cuda.is_available() else"cpu"
dtype = torch.float16 # Use float16 to save VRAM on your GPU

# 2. Use the official Hugging Face ID instead of a local SD_PATH
# This guarantees all files (like vocab.json) download perfectly
HF_ID ="runwayml/stable-diffusion-v1-5"

print("Loading SD v1.5 components...")

# 3. Load base components using the HF_ID
tokenizer = CLIPTokenizer.from_pretrained(HF_ID, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(HF_ID, subfolder="text_encoder", torch_dtype=dtype).to(device)
vae = AutoencoderKL.from_pretrained(HF_ID, subfolder="vae", torch_dtype=dtype).to(device)
unet = UNet2DConditionModel.from_pretrained(HF_ID, subfolder="unet", torch_dtype=dtype).to(device)
noise_scheduler = DDPMScheduler.from_pretrained(HF_ID, subfolder="scheduler")

print("Loading ControlNet models...")

# Canny ControlNet
controlnet_canny = ControlNetModel.from_pretrained(
"lllyasviel/sd-controlnet-canny",
 torch_dtype=dtype
).to(device)

# Segmentation ControlNet
controlnet_seg = ControlNetModel.from_pretrained(
"lllyasviel/sd-controlnet-seg",
 torch_dtype=dtype
).to(device)

print("Models loaded successfully.")

Loading SD v1.5 components...
Loading ControlNet models...
✓ Models loaded successfully.


## Cell 7: Freeze Base Models & Inject LoRA

In [13]:
# Freeze everything
text_encoder.requires_grad_(False)
vae.requires_grad_(False)
unet.requires_grad_(False)
controlnet_canny.requires_grad_(False)
controlnet_seg.requires_grad_(False)

# Inject LoRA into UNet attention layers
lora_config = LoraConfig(
 r=4,
 lora_alpha=32,
 target_modules=["to_k","to_q","to_v","to_out.0"],
 lora_dropout=0.1,
 bias="none",
)

unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()

# Enable gradient checkpointing
unet.enable_gradient_checkpointing()

print("LoRA injected into UNet.")

trainable params: 797,184 || all params: 860,318,148 || trainable%: 0.0927
LoRA injected into UNet.


## Cell 8: Create Dataset & DataLoader

In [14]:
print("Creating training dataset...")
train_dataset = PennCannySegDataset(
 csv_path=TRAIN_CSV_PATH,
 image_dir=IMAGE_DIR,
 canny_dir=CANNY_MAP_DIR,
 seg_dir=SEG_MAP_DIR,
 tokenizer=tokenizer,
 resolution=512,
)

print("\nCreating validation dataset...")
val_dataset = PennCannySegDataset(
 csv_path=VAL_CSV_PATH,
 image_dir=IMAGE_DIR,
 canny_dir=CANNY_MAP_DIR,
 seg_dir=SEG_MAP_DIR,
 tokenizer=tokenizer,
 resolution=512,
)

train_dataloader = DataLoader(
 train_dataset,
 batch_size=4,
 shuffle=True,
 num_workers=2,
 collate_fn=collate_fn,
)

val_dataloader = DataLoader(
 val_dataset,
 batch_size=4,
 shuffle=False,
 num_workers=2,
 collate_fn=collate_fn,
)

print(f"\nTrain samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

# Sanity check
sample = train_dataset[0]
print(f"\nSample check:")
print(f"Image shape: {sample['pixel_values'].shape}")
print(f"Canny shape: {sample['canny_values'].shape}")
print(f"Seg shape: {sample['seg_values'].shape}")
print(f"Prompt: {sample['prompt'][:80]}...")

Creating training dataset...
Skipping missing file: towne_steps_night_cloudy_frontmid.jpg
Dataset initialized with 244 valid samples

Creating validation dataset...
Dataset initialized with 62 valid samples

Train samples: 244
Validation samples: 62

Sample check:
Image shape: torch.Size([3, 512, 512])
Canny shape: torch.Size([3, 512, 512])
Seg shape: torch.Size([3, 512, 512])
Prompt: Skirkanich building modern engineering facility, glass and metal construction, c...


## Cell 9: Training Configuration

In [15]:
# Training hyperparameters
NUM_EPOCHS = 15
LEARNING_RATE = 1e-4
SAVE_EVERY_N_EPOCHS = 5
VALIDATE_EVERY_N_EPOCHS = 5

# Optimization
GRAD_ACCUM_STEPS = 1
MAX_GRAD_NORM = 1.0

# Early stopping
PATIENCE = 5
MIN_DELTA = 0.001

# A100 optimizations
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Optimizer (only LoRA params)
trainable_params = [p for p in unet.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE)

print(f"Training config set. {len(trainable_params)} param groups.")

Training config set. 256 param groups.


## Cell 10: Validation Function

In [16]:
def compute_validation_loss(unet, val_dataloader, vae, text_encoder, noise_scheduler, controlnet_canny, controlnet_seg):
"""Compute validation loss on held-out set."""
 unet.eval()
 val_loss = 0

 with torch.no_grad():
 for batch in val_dataloader:
 pixel_values = batch["pixel_values"].to(device, dtype=dtype)
 canny_values = batch["canny_values"].to(device, dtype=dtype)
 seg_values = batch["seg_values"].to(device, dtype=dtype)

 input_ids = batch["input_ids"].to(device)

 # VAE encoding
 latents = vae.encode(pixel_values).latent_dist.sample()
 latents = latents * vae.config.scaling_factor

 # Sample noise
 noise = torch.randn_like(latents)
 bsz = latents.shape[0]
 timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (bsz,), device=device).long()
 noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

 # Text encoding
 encoder_hidden_states = text_encoder(input_ids)[0]

 # ControlNet conditioning
 down_block_res_samples_canny, mid_block_res_sample_canny = controlnet_canny(
 noisy_latents, timesteps, encoder_hidden_states=encoder_hidden_states,
 controlnet_cond=canny_values, return_dict=False,
 )
 down_block_res_samples_seg, mid_block_res_sample_seg = controlnet_seg(
 noisy_latents, timesteps, encoder_hidden_states=encoder_hidden_states,
 controlnet_cond=seg_values, return_dict=False,
 )

 down_block_res_samples = [
 canny * 0.8 + seg * 0.6
 for canny, seg in zip(down_block_res_samples_canny, down_block_res_samples_seg)
 ]
 mid_block_res_sample = mid_block_res_sample_canny * 0.8 + mid_block_res_sample_seg * 0.6

 # UNet forward
 with torch.autocast("cuda", dtype=torch.bfloat16):
 noise_pred = unet(
 noisy_latents, timesteps, encoder_hidden_states,
 down_block_additional_residuals=down_block_res_samples,
 mid_block_additional_residual=mid_block_res_sample,
 ).sample

 loss = F.mse_loss(noise_pred.float(), noise.float(), reduction="mean")
 val_loss += loss.item()

 unet.train()
 return val_loss / len(val_dataloader)

print("Validation loss function defined.")

Validation loss function defined.


In [17]:
def run_validation_generation(unet, vae, text_encoder, tokenizer, controlnet_canny, controlnet_seg, epoch, num_samples=3):
"""Generate validation images from random validation samples."""
 print(f"\nGenerating validation images at epoch {epoch}...")

 # Build img2img pipeline
 pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
 SD_PATH,
 unet=unet,
 vae=vae,
 text_encoder=text_encoder,
 tokenizer=tokenizer,
 controlnet=[controlnet_canny, controlnet_seg],
 torch_dtype=torch.float16,
 safety_checker=None,
 ).to(device)

 pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
 pipe.set_progress_bar_config(disable=True)

 unet.eval()

 # Sample random validation examples
 import random
 val_samples = random.sample(range(len(val_dataset)), min(num_samples, len(val_dataset)))

 for i, idx in enumerate(val_samples):
 sample = val_dataset[idx]
 filename = val_dataset.df.iloc[idx]['filename']

 source_image = Image.open(os.path.join(IMAGE_DIR, filename)).convert("RGB").resize((512, 512))
 canny_map = Image.open(os.path.join(CANNY_MAP_DIR, filename)).convert("RGB").resize((512, 512))
 seg_map = Image.open(os.path.join(SEG_MAP_DIR, filename)).convert("RGB").resize((512, 512))

 with torch.no_grad():
 generated = pipe(
 prompt=sample["prompt"],
 image=source_image,
 control_image=[canny_map, seg_map],
 strength=0.4,
 num_inference_steps=30,
 guidance_scale=7.5,
 controlnet_conditioning_scale=[0.8, 0.6],
 ).images[0]

 output_path = os.path.join(VALIDATION_DIR, f"epoch_{epoch}_sample_{i}_{filename}")
 generated.save(output_path)
 print(f"Saved: {output_path}")

 unet.train()
 del pipe
 torch.cuda.empty_cache()

print("Validation generation function defined.")

Validation generation function defined.


## Cell 11: Training Loop with Canny + Segmentation

In [18]:
unet.train()
global_step = 0
best_val_loss = float('inf')
patience_counter = 0
best_epoch = 0

for epoch in range(NUM_EPOCHS):
 # Training phase
 epoch_train_loss = 0
 progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")

 for step, batch in enumerate(progress_bar):
 pixel_values = batch["pixel_values"].to(device, dtype=dtype)
 canny_values = batch["canny_values"].to(device, dtype=dtype)
 seg_values = batch["seg_values"].to(device, dtype=dtype)
 input_ids = batch["input_ids"].to(device)

 # VAE encoding
 with torch.no_grad():
 latents = vae.encode(pixel_values).latent_dist.sample()
 latents = latents * vae.config.scaling_factor

 # Sample noise and timesteps
 noise = torch.randn_like(latents)
 bsz = latents.shape[0]
 timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (bsz,), device=device).long()
 noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

 # Text encoding
 with torch.no_grad():
 encoder_hidden_states = text_encoder(input_ids)[0]

 # ControlNet conditioning
 with torch.no_grad():
 down_block_res_samples_canny, mid_block_res_sample_canny = controlnet_canny(
 noisy_latents, timesteps, encoder_hidden_states=encoder_hidden_states,
 controlnet_cond=canny_values, return_dict=False,
 )
 down_block_res_samples_seg, mid_block_res_sample_seg = controlnet_seg(
 noisy_latents, timesteps, encoder_hidden_states=encoder_hidden_states,
 controlnet_cond=seg_values, return_dict=False,
 )

 down_block_res_samples = [
 canny * 0.8 + seg * 0.6
 for canny, seg in zip(down_block_res_samples_canny, down_block_res_samples_seg)
 ]
 mid_block_res_sample = mid_block_res_sample_canny * 0.8 + mid_block_res_sample_seg * 0.6

 # UNet forward in bfloat16
 with torch.autocast("cuda", dtype=torch.bfloat16):
 noise_pred = unet(
 noisy_latents, timesteps, encoder_hidden_states,
 down_block_additional_residuals=down_block_res_samples,
 mid_block_additional_residual=mid_block_res_sample,
 ).sample

 loss = F.mse_loss(noise_pred.float(), noise.float(), reduction="mean")
 loss = loss / GRAD_ACCUM_STEPS

 # Backprop
 loss.backward()

 # Gradient accumulation
 if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_dataloader):
 torch.nn.utils.clip_grad_norm_(unet.parameters(), MAX_GRAD_NORM)
 optimizer.step()
 optimizer.zero_grad(set_to_none=True)
 global_step += 1

 true_loss = loss.item() * GRAD_ACCUM_STEPS
 epoch_train_loss += true_loss
 progress_bar.set_postfix({"train_loss": true_loss})

 avg_train_loss = epoch_train_loss / len(train_dataloader)

 # Validation phase
 print(f"\nComputing validation loss...")
 val_loss = compute_validation_loss(
 unet, val_dataloader, vae, text_encoder, noise_scheduler,
 controlnet_canny, controlnet_seg
 )

 print(f"Epoch {epoch+1}: train_loss={avg_train_loss:.4f}, val_loss={val_loss:.4f}")

 # Early stopping based on validation loss
 if val_loss < best_val_loss - MIN_DELTA:
 best_val_loss = val_loss
 best_epoch = epoch + 1
 patience_counter = 0
 best_path = os.path.join(LORA_OUTPUT_DIR,"lora_best")
 unet.save_pretrained(best_path)
 print(f"New best validation loss — saved to {best_path}")
 else:
 patience_counter += 1
 print(f"No improvement for {patience_counter}/{PATIENCE} epochs")

 # Checkpointing
 if (epoch + 1) % SAVE_EVERY_N_EPOCHS == 0:
 save_path = os.path.join(LORA_OUTPUT_DIR, f"lora_epoch_{epoch+1}")
 unet.save_pretrained(save_path)
 print(f"Saved checkpoint to {save_path}")

 # Validation image generation
 if (epoch + 1) % VALIDATE_EVERY_N_EPOCHS == 0:
 run_validation_generation(
 unet, vae, text_encoder, tokenizer,
 controlnet_canny, controlnet_seg, epoch + 1
 )

 # Early stopping trigger
 if patience_counter >= PATIENCE:
 print(f"\n Early stopping at epoch {epoch+1}")
 print(f"Best validation loss: {best_val_loss:.4f} at epoch {best_epoch}")
 break

# Final save
final_path = os.path.join(LORA_OUTPUT_DIR,"lora_final")
unet.save_pretrained(final_path)
print(f"\n Training complete!")
print(f"Best checkpoint: lora_best (epoch {best_epoch}, val_loss={best_val_loss:.4f})")

Epoch 1/15:   0%|          | 0/61 [00:00<?, ?it/s]


Computing validation loss...
Epoch 1: train_loss=0.1363, val_loss=0.1212
✓ New best validation loss — saved to /content/drive/MyDrive/5190 S26 Final Project/Dataset/LoRA_CannySegmentation_Checkpoints/lora_best


Epoch 2/15:   0%|          | 0/61 [00:00<?, ?it/s]


Computing validation loss...
Epoch 2: train_loss=0.1396, val_loss=0.1082
✓ New best validation loss — saved to /content/drive/MyDrive/5190 S26 Final Project/Dataset/LoRA_CannySegmentation_Checkpoints/lora_best


Epoch 3/15:   0%|          | 0/61 [00:00<?, ?it/s]

...

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.controlnet.pipeline_controlnet_img2img.StableDiffusionControlNetImg2ImgPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['clouds, university of pennsylvania campus architecture, philadelphia collegiate setting, professional architectural photography, detailed structural rendering, upenn']
The following part of your input was truncated because CLIP ca

Saved: /content/drive/MyDrive/5190 S26 Final Project/Validation_CannySegmentation/epoch_15_sample_0_hayden_hall_night_cloudy_sideview3.jpg


The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['colors, crisp edges and details, high dynamic range lighting conditions, university of pennsylvania campus architecture, philadelphia collegiate setting, professional architectural photography, detailed structural rendering, upenn']


Saved: /content/drive/MyDrive/5190 S26 Final Project/Validation_CannySegmentation/epoch_15_sample_1_chem_lab_side_front_day_sunny.jpg
Saved: /content/drive/MyDrive/5190 S26 Final Project/Validation_CannySegmentation/epoch_15_sample_2_hayden_hall_slope_day_sunny_fromTowne.jpg

✓ Training complete!
Best checkpoint: lora_best (epoch 12, val_loss=0.0742)


## Training Complete!

### What You Have Now
1. **Canny maps** — fine architectural detail edges (from preprocessing)
2. **Segmentation maps** — semantic region labels (just generated)
3. **Trained LoRA** — Penn-specific adapter working with Canny + Segmentation
4. **Checkpoints** — saved every 5 epochs + best model
5. **Validation outputs** — visual quality check across training

### Why This Architecture Works
- **Canny:** Preserves window grids, doorframes, brick patterns, ornamental details
- **Segmentation:** Prevents sky→building hallucinations, keeps semantic regions correct
- **Together:** Geometric fidelity + semantic correctness = best LPIPS on architecture

### Next Steps
1. Check `Validation_CannySegmentation/` folder — compare across epochs
2. Pick best checkpoint visually (not just by loss)
3. Build inference pipeline (next cell)
4. Generate test outputs
5. Compute LPIPS vs ground truth
6. Compare with other architectures (Canny+Depth, Seg+Depth)

### Tuning Parameters at Inference
- `strength` (0.3-0.5): how much to change from source
- `controlnet_conditioning_scale` ([0.6-1.0, 0.4-0.8]): Canny vs Seg influence
 - Higher Canny = more fine detail
 - Higher Seg = stronger semantic constraints
- `guidance_scale` (7-9): prompt adherence